# The Invisible Half — Rwanda Vital Statistics 2019–2025
**NISR 2026 Infographic Competition · Mathematics for Machine Learning (Cohort 3)**

Rwanda records almost every birth (92.9% completeness in 2025) but only about half of its deaths (50.5%).
This notebook loads the official NISR CRVS 2025 tables, checks them, and uses five methods to answer
*how fast is the gap closing, and who is still missing?*

1. Linear regression with the normal equation
2. A logistic S-curve fitted by gradient descent (from scratch)
3. Derivatives: how fast completeness is rising
4. Train/test validation: line vs S-curve
5. Probability: what are the chances a death is recorded?

It then builds the figures, the static poster and the dynamic (Plotly) infographic.

**Source:** NISR, *Rwanda Vital Statistics Report 2025*, `Final Tables_CRVS_2025.xlsx`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyBboxPatch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PATH = "../data/Final Tables_CRVS_2025_0.xlsx"
FIG = Path("../figures")
FIG.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# One palette for every chart, the poster and the animation
INK, MUTED, GRID, PAPER = "#1F2A33", "#6F7A83", "#E2DDD3", "#FAF7F2"
DEATH, BIRTH, ACCENT = "#C4502F", "#2B7A8C", "#E09A2D"

plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "font.size": 11, "axes.titleweight": "bold",
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.axisbelow": True,
})

## 1. Load and check the data

### 1.1 What is in the file

In [ ]:
xl = pd.ExcelFile(PATH)
print(len(xl.sheet_names), "sheets")
# repr shows hidden spaces: 'Figure7 ' has a trailing space
print([s for s in xl.sheet_names if s != s.strip()])

In [ ]:
def peek(sheet, rows=11):
    df = pd.read_excel(PATH, sheet_name=sheet, header=None)
    print(f"--- {sheet}: shape {df.shape} ---")
    return df.head(rows)

peek("Table 14")

### 1.2 Clean tables
Every NISR sheet has an empty first column, a title row and a source row, so each load picks the
header row (`header`), the useful columns (`usecols`) and the data rows (`nrows`).

In [ ]:
# 10.6 — birth and death registration completeness (%), 2019–2025
completeness = pd.read_excel(PATH, sheet_name="10.6", header=2, usecols=[1, 2, 3], nrows=7)
completeness.columns = ["year", "birth", "death"]
completeness

In [ ]:
# Table 12 — mortality summary; indicators are rows and years are columns
t12 = pd.read_excel(PATH, sheet_name="Table 12", header=2, usecols=range(1, 9), nrows=12)
t12 = t12.set_index("Indicator")
# "Male"/"Female" appear three times, so give every row a unique name before transposing
t12.index = ["registered", "reg_male", "reg_female",
             "expected", "exp_male", "exp_female",
             "completeness", "comp_male", "comp_female",
             "cdr", "u5mr", "sex_ratio"]
mortality = t12.T
mortality.index = mortality.index.astype(int)
mortality.index.name = "year"
mortality

In [ ]:
# Table 14 — registered deaths by place of occurrence, 30 districts (2025)
districts = pd.read_excel(PATH, sheet_name="Table 14", header=None, skiprows=4, nrows=30,
                          usecols=[1, 2, 3, 4, 5])
districts.columns = ["district", "community", "facility", "total", "community_pct_nisr"]
districts["district"] = districts["district"].str.strip()
# recompute the share from the counts and check it against NISR's own percentage
districts["community_pct"] = districts["community"] / districts["total"] * 100
districts.sort_values("community_pct", ascending=False).head()

In [ ]:
# Table 16 — verbal autopsies for deaths that occurred in the reporting year
va = pd.read_excel(PATH, sheet_name="Table 16", header=2, usecols=range(1, 6), nrows=1)
va = va.set_index("Indicator").T
va.index = va.index.astype(int)
va.columns = ["verbal_autopsies"]

# Table 11 — deaths registered late (backlog), header is on row 3 in this sheet
t11 = pd.read_excel(PATH, sheet_name="Table 11", header=3, usecols=range(1, 6), nrows=4)
t11 = t11.set_index(t11.columns[0])  # this header cell is "Indicator " with a hidden space
t11.index = ["same_year", "one_year_late", "over_one_year_late", "total"]
backlog = t11.T
backlog.index = backlog.index.astype(int)

# Figure 17 — registered and unregistered deaths as shares of expected deaths
f17 = pd.read_excel(PATH, sheet_name="Figure 17", header=2, usecols=range(1, 9), nrows=3)
f17 = f17.set_index(f17.columns[0])
f17.index = ["timely", "late", "unregistered"]
shares = f17.T
shares.index = shares.index.astype(int)

# Figure 21 — community vs facility deaths by rural/urban residence (2025)
residence = pd.read_excel(PATH, sheet_name="Figure 21", header=2, usecols=range(1, 7), nrows=2)
residence.columns = ["area", "community", "facility", "total", "community_share", "facility_share"]

display(va.T, backlog, shares.T, residence)

### 1.3 Accuracy checks and known inconsistencies
Accuracy is judged, so the values used are checked against the brief and the source's internal
inconsistencies are handled explicitly:

| Issue | Values in the source | Used |
|---|---|---|
| Deaths 2025 | 39,378 (sheet 10.2) vs 39,355 (Tables 12/14, 10.5) | **39,355** |
| Deaths 2022 | 25,567 (10.2, Table 11) vs 25,536 (Table 12) | **25,536** |
| Female death completeness 2025 | 43.6 (Table 12) vs 43.7 (sheet 10.5) | **43.7**, recomputed from Table 12 counts: 17,304 / 39,614 |
| Expected deaths 2025 | 77,876 (Table 12) vs 77,875 (10.5) | **77,876** |
| Under-5 mortality | Table 12 and sheet 10.8 disagree (e.g. 32.7 vs 35.7 in 2025) | not used |
| Usable community causes of death 2025 | 16.1% (Fig 23) vs 83.9% (Fig 24) | not used |
| Place of death | Table 14 says *community*, not *home* | labelled "outside a health facility (community)" |

**Rule on the poster: where tables differ, Table 12 values are used.**

In [ ]:
# Female completeness: Table 12 prints 43.6, its own counts give 43.7
female_2025 = mortality.loc[2025, "reg_female"] / mortality.loc[2025, "exp_female"] * 100
print(f"female completeness from counts: {female_2025:.2f}%  (Table 12 prints {mortality.loc[2025, 'comp_female']})")
mortality.loc[2025, "comp_female"] = round(female_2025, 1)

# Every completeness value should equal registered / expected
recomputed = (mortality["registered"] / mortality["expected"] * 100).round(1)
print("max gap between printed and recomputed completeness:", (recomputed - mortality["completeness"]).abs().max())

assert list(completeness["death"]) == [31.4, 29.9, 26.2, 31.1, 41.8, 46.1, 50.5]
assert list(completeness["birth"]) == [87.0, 85.8, 84.2, 92.9, 90.0, 90.3, 92.9]
assert (completeness.set_index("year")["death"] == mortality["completeness"]).all()
assert mortality.loc[2025, "registered"] == 39_355 and mortality.loc[2022, "registered"] == 25_536
assert mortality.loc[2019, "registered"] == 23_791
assert mortality.loc[2025, "comp_male"] == 57.6 and mortality.loc[2025, "comp_female"] == 43.7
assert len(districts) == 30 and districts["total"].sum() + 293 + 15 == 39_355  # + "not stated" + "abroad"
assert (districts["community_pct"] - districts["community_pct_nisr"]).abs().max() < 0.06
assert list(va["verbal_autopsies"]) == [1889, 8369, 13703, 18921]
print("all checks passed")

### 1.4 First look
Births have been recorded at 84–93% every year. Deaths fell to 26% in 2021 and then climbed every year
to 50.5% in 2025. Note that expected deaths jump to 82,241 in 2022 (the female estimate rises from about
36,900 to 42,950), so part of the 2022 value reflects a change in the denominator, not in registration.

In [ ]:
years = completeness["year"].to_numpy()
y_death = completeness["death"].to_numpy()
y_birth = completeness["birth"].to_numpy()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(years, y_birth, "o-", color=BIRTH, lw=2.5, label="Births")
ax.plot(years, y_death, "o-", color=DEATH, lw=2.5, label="Deaths")
for x, b, d in zip(years, y_birth, y_death):
    ax.annotate(f"{b:.0f}", (x, b), textcoords="offset points", xytext=(0, 8), ha="center", color=BIRTH, fontsize=9)
    ax.annotate(f"{d:.1f}", (x, d), textcoords="offset points", xytext=(0, -16), ha="center", color=DEATH, fontsize=9)
ax.set_ylim(0, 100)
ax.set_ylabel("Registration completeness (%)")
ax.set_title("Births are almost all recorded; deaths only about half")
ax.legend(frameon=False, loc="center right")
fig.tight_layout()
fig.savefig(FIG / "01_trend.png", dpi=200)
plt.show()

mortality[["registered", "expected", "completeness"]].assign(
    unrecorded=mortality["expected"] - mortality["registered"])

## 2. Linear regression with the normal equation

Model: $y = \theta_0 + \theta_1 t$, with $t$ = years since 2019. With the design matrix
$X = [\mathbf{1}, t]$, least squares gives

$$\theta = (X^\top X)^{-1} X^\top y$$

We compare two data windows: all seven years, and 2021–2025 (the recovery after the 2020–21 COVID-19 years).

In [ ]:
BASE = 2019  # t = year - 2019 keeps the numbers small and well-conditioned

def design(t):
    return np.column_stack([np.ones_like(t, dtype=float), t])

def normal_equation(t, y):
    X = design(t)
    # solve (XᵀX) θ = Xᵀy instead of forming the inverse: same maths, numerically safer
    return np.linalg.solve(X.T @ X, X.T @ y)

def year_reaching(level, theta):
    return BASE + (level - theta[0]) / theta[1]

t_all = (years - BASE).astype(float)
windows = {"2019–2025 (all years)": years >= 2019, "2021–2025 (recovery)": years >= 2021}

lines = {}
for name, mask in windows.items():
    theta = normal_equation(t_all[mask], y_death[mask])
    assert np.allclose(theta[::-1], np.polyfit(t_all[mask], y_death[mask], 1))  # sanity check
    lines[name] = theta
    print(f"{name}: slope = {theta[1]:+.2f} pts/yr, 90% in {year_reaching(90, theta):.1f}, "
          f"100% in {year_reaching(100, theta):.1f}")

The data window matters: using all years the line rises **+3.8 points a year** (90% around 2036);
using 2021–2025 it rises **+6.4 points a year** (90% around 2031), a difference of about five years.
A straight line also passes 100% (around 2033 for the recovery window), which is impossible for a
percentage. That is why we need a model with a ceiling.

## 3. Logistic S-curve fitted by gradient descent

$$\hat y(t) = \frac{100}{1 + e^{-k(t - t_0)}}$$

$k$ is the speed of the catch-up and $t_0$ is the year completeness crosses 50%.
Loss (mean squared error): $L = \frac{1}{n}\sum_i(\hat y_i - y_i)^2$.

With $s_i = 1/(1 + e^{-k(t_i - t_0)})$, so $\hat y_i = 100 s_i$, the chain rule and $s' = s(1-s)$ give

$$\frac{\partial L}{\partial k} = \frac{1}{n}\sum_i 2(\hat y_i - y_i)\cdot 100\, s_i(1-s_i)\,(t_i - t_0),\qquad
\frac{\partial L}{\partial t_0} = \frac{1}{n}\sum_i 2(\hat y_i - y_i)\cdot 100\, s_i(1-s_i)\,(-k)$$

Update: $\theta \leftarrow \theta - \alpha \nabla L$. Because $k$ (≈0.3) and $t_0$ (≈5) live on very
different scales, each gets its own learning rate.

**Why MSE?** If the errors around the curve are Normal with constant variance, minimising MSE is the same as
maximising the likelihood, so the MSE fit is the maximum-likelihood fit.

In [ ]:
def logistic(t, k, t0):
    return 100 / (1 + np.exp(-k * (t - t0)))

def gradients(t, y, k, t0):
    s = 1 / (1 + np.exp(-k * (t - t0)))
    err = 100 * s - y
    common = 2 * err * 100 * s * (1 - s)
    return np.mean(err ** 2), np.mean(common * (t - t0)), np.mean(common * (-k))

def gradient_descent(t, y, k=0.5, t0=4.0, lr_k=5e-5, lr_t0=5e-3, iters=20_000, keep=()):
    history, snapshots = [], {}
    for i in range(iters):
        loss, dk, dt0 = gradients(t, y, k, t0)
        history.append(loss)
        if i in keep:
            snapshots[i] = (k, t0, loss)
        k, t0 = k - lr_k * dk, t0 - lr_t0 * dt0
    return k, t0, np.array(history), snapshots

def logit_start(t, y):
    # log(y / (100 - y)) = k·t - k·t0 is a straight line, so the normal equation gives a starting point
    theta = normal_equation(t, np.log(y / (100 - y)))
    return theta[1], -theta[0] / theta[1]

def year_reaching_s(level, k, t0):
    # invert the curve: t = t0 + ln(level / (100 - level)) / k
    return BASE + t0 + np.log(level / (100 - level)) / k

In [ ]:
recovery = years >= 2021
t_fit, y_fit = t_all[recovery], y_death[recovery]

# naive start vs the logit (normal-equation) start: both should land on the same minimum
SNAP = [0, 1, 2, 3, 5, 8, 12, 20, 35, 60, 100, 200, 400, 1000, 3000, 19_999]
k, t0, loss_hist, snaps = gradient_descent(t_fit, y_fit, k=0.5, t0=4.0, keep=SNAP)
k_l, t0_l = logit_start(t_fit, y_fit)
k_b, t0_b, _, _ = gradient_descent(t_fit, y_fit, k=k_l, t0=t0_l)

print(f"naive start  -> k = {k:.4f}, 50% crossed in {BASE + t0:.2f}, final MSE = {loss_hist[-1]:.3f}")
print(f"logit start  -> k = {k_b:.4f}, 50% crossed in {BASE + t0_b:.2f}")
print(f"S-curve reaches 90% in {year_reaching_s(90, k, t0):.1f}")
assert abs(k - k_b) < 1e-3 and abs(t0 - t0_b) < 1e-2

k_all, t0_all, _, _ = gradient_descent(t_all, y_death, *logit_start(t_all, y_death))
print(f"(all 7 years, for comparison: k = {k_all:.3f}, 90% in {year_reaching_s(90, k_all, t0_all):.1f})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_hist, color=DEATH, lw=2)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Iteration")
ax.set_ylabel("MSE loss (pts²)")
ax.set_title(f"Gradient descent converges: MSE {loss_hist[0]:.0f} → {loss_hist[-1]:.1f}")
fig.tight_layout()
fig.savefig(FIG / "02_loss_curve.png", dpi=200)
plt.show()

In [ ]:
t_future = np.linspace(0, 16, 400)  # 2019–2035
theta_rec = lines["2021–2025 (recovery)"]
line_future = theta_rec[0] + theta_rec[1] * t_future

fig, ax = plt.subplots(figsize=(10, 5))
ax.axvspan(2020, 2021, color=GRID, alpha=0.6, lw=0)
ax.text(2020.5, 96, "COVID-19\nyears", ha="center", va="top", color=MUTED, fontsize=9)
ax.axhline(90, color=MUTED, lw=1, ls=":")
ax.plot(BASE + t_future[t_future >= 2], logistic(t_future[t_future >= 2], k, t0), color=DEATH, lw=2, ls="--",
        label="S-curve (gradient descent, 2021–25)")
ok = line_future <= 100
ax.plot(BASE + t_future[ok], line_future[ok], color=MUTED, lw=1.5, ls="-.", label="Straight line (2021–25)")
ax.plot(years, y_birth, "o-", color=BIRTH, lw=2.5, label="Births recorded")
ax.plot(years, y_death, "o-", color=DEATH, lw=2.5, label="Deaths recorded")
ax.set_xlim(2018.5, 2035.5)
ax.set_ylim(0, 102)
ax.set_ylabel("Registration completeness (%)")
ax.set_title(f"If the 2021–25 pace continues, 90% of deaths recorded around "
             f"{year_reaching(90, theta_rec):.0f}–{year_reaching_s(90, k, t0):.0f}")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "03_projection.png", dpi=200)
plt.show()

## 4. Derivative: how fast is completeness rising?

Differentiating the logistic curve gives

$$\frac{dy}{dt} = k\,y\left(1 - \frac{y}{100}\right)$$

This is largest when $y = 50$ (set $\frac{d}{dy}[y(1-y/100)] = 1 - 2y/100 = 0$), where the rate is $25k$.
Rwanda reached 50.5% in 2025, so if the model holds it is at the **fastest point** of its catch-up:
each further point will come a little more slowly.

In [ ]:
def rate(y_level, k):
    return k * y_level * (1 - y_level / 100)

t_grid = np.linspace(-2, 21, 500)  # 2017–2040
y_grid = logistic(t_grid, k, t0)
print(f"peak rate = 25k = {25 * k:.2f} pts/yr, reached in {BASE + t0:.1f}")
print(f"rate at 2025's 50.5% = {rate(50.5, k):.2f} pts/yr;  at 80% = {rate(80, k):.2f};  at 90% = {rate(90, k):.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(BASE + t_grid, rate(y_grid, k), color=DEATH, lw=2.5)
ax.axvline(BASE + t0, color=MUTED, ls=":", lw=1)
ax.annotate(f"Fastest: {25 * k:.1f} pts/yr\nat 50% ({BASE + t0:.1f})", (BASE + t0, 25 * k),
            xytext=(12, -30), textcoords="offset points", color=INK, fontsize=10)
ax.set_ylabel("Gain in completeness (pts / year)")
ax.set_title("The catch-up is at its fastest now; later gains get harder")
fig.tight_layout()
fig.savefig(FIG / "04_derivative.png", dpi=200)
plt.show()

## 5. Validation: train on the past, test on 2024–2025

We hide 2024 and 2025, fit both models on the earlier years, and measure the error on the hidden years
(RMSE, in percentage points). Two training windows are tested because the 2019–2021 decline is
very different from the recovery that follows.

In [ ]:
test = years >= 2024
rows = []
for name, train in {"2019–2023": years <= 2023, "2021–2023": (years >= 2021) & (years <= 2023)}.items():
    th = normal_equation(t_all[train], y_death[train])
    pred_line = design(t_all[test]) @ th
    ks, t0s, _, _ = gradient_descent(t_all[train], y_death[train], *logit_start(t_all[train], y_death[train]))
    pred_s = logistic(t_all[test], ks, t0s)
    for model, pred in [("Straight line", pred_line), ("S-curve", pred_s)]:
        rows.append({"train on": name, "model": model,
                     "pred 2024": pred[0], "pred 2025": pred[1],
                     "RMSE (pts)": np.sqrt(np.mean((pred - y_death[test]) ** 2))})
validation = pd.DataFrame(rows).round(1)
print("actual: 2024 = 46.1, 2025 = 50.5")
validation

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
labels = [f"{r['model']}\n(train {r['train on']})" for _, r in validation.iterrows()]
colors = [MUTED if m == "Straight line" else DEATH for m in validation["model"]]
bars = ax.barh(labels, validation["RMSE (pts)"], color=colors)
ax.bar_label(bars, fmt="%.1f pts", padding=4)
ax.invert_yaxis()
ax.set_xlabel("Error on hidden years 2024–2025 (RMSE, pts)")
ax.set_title("Test error on 2024–25: the training window matters most")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(FIG / "05_validation.png", dpi=200)
plt.show()

**Reading the result.** Trained on 2019–2023, both models miss 2024–25 badly (7–9 points) because the
COVID-19 dip pulls the fit down; the S-curve is a little better. Trained on the recovery years only, both
do far better, and the straight line is slightly more accurate over this two-year horizon.

**Model choice.** Over two years the two models are almost equally good, but the straight line passes 100%
around 2033, which is impossible, so the S-curve is the one used for long-range projection. The poster reports
the honest range: **90% of deaths recorded around 2031 (line) to 2033 (S-curve), if the 2021–25 trend continues.**
With only 7 data points these are scenarios, not forecasts.

## 6. Probability: what are the chances a death is recorded?

Treat each expected death as a trial that is either recorded or not, with probability $p$.
For $k$ recorded out of $n$ expected, the binomial likelihood $L(p) = \binom{n}{k}p^k(1-p)^{n-k}$ is
maximised at $\hat p = k/n$, which is exactly NISR's completeness rate. (Expected deaths are themselves
estimates, so this is an approximation.)

In [ ]:
m25 = mortality.loc[2025]
n, k_rec = m25["expected"], m25["registered"]
p = k_rec / n
se = np.sqrt(p * (1 - p) / n)
unrecorded = n - k_rec

p_m = m25["reg_male"] / m25["exp_male"]
p_f = m25["reg_female"] / m25["exp_female"]
se_diff = np.sqrt(p_m * (1 - p_m) / m25["exp_male"] + p_f * (1 - p_f) / m25["exp_female"])
z = (p_m - p_f) / se_diff

# Bayes: P(man | recorded) = P(recorded | man) P(man) / P(recorded)
p_man = m25["exp_male"] / n
p_man_given_rec = p_m * p_man / p

print(f"P(death recorded)               = {p:.3f}  (95% CI ±{1.96 * se * 100:.2f} pts)")
print(f"Unrecorded deaths in 2025       = {unrecorded:,.0f}  (about {unrecorded / 365:.0f} a day)")
print(f"P(recorded | man)               = {p_m:.3f}")
print(f"P(recorded | woman)             = {p_f:.3f}")
print(f"gap = {100 * (p_m - p_f):.1f} pts, z = {z:.0f}  -> not chance")
print(f"P(man) among all expected deaths = {p_man:.3f}")
print(f"P(man | death recorded)          = {p_man_given_rec:.3f}  (Bayes)")
print(f"Expected recorded out of 10 deaths: men {10 * p_m:.1f}, women {10 * p_f:.1f}, all {10 * p:.1f}")

The sampling uncertainty is tiny (about ±0.35 points), because $n$ is large. The real uncertainty is in the
model and in the short series, not in sampling. The gender gap is about 39 standard errors wide, so it is not
chance. By Bayes' theorem, men are **49% of all deaths but 56% of recorded deaths**, so statistics built from
registered deaths over-represent men.

In [ ]:
def person(ax, x, y, h, color):
    # a simple person icon: head + rounded body, (x, y) is the bottom centre
    ax.add_patch(Circle((x, y + 0.80 * h), 0.15 * h, color=color, lw=0))
    ax.add_patch(FancyBboxPatch((x - 0.18 * h, y), 0.36 * h, 0.58 * h,
                                boxstyle=f"round,pad=0,rounding_size={0.12 * h}", color=color, lw=0))

def icon_row(ax, filled, n=10, color=DEATH, empty=GRID, y=0.0, h=1.0, gap=0.55):
    for i in range(n):
        person(ax, i * gap * h * 1.0 + 0.2, y, h * 0.9, color if i < filled else empty)
    ax.set_xlim(-0.2, n * gap * h + 0.1)
    ax.set_aspect("equal")
    ax.axis("off")

fig, axes = plt.subplots(3, 1, figsize=(7, 4.6))
for ax, (label, prob, color) in zip(axes, [("All deaths", p, DEATH), ("Men", p_m, DEATH), ("Women", p_f, DEATH)]):
    icon_row(ax, round(10 * prob), color=color)
    ax.set_ylim(-0.1, 1.0)
    ax.set_title(f"{label}: {100 * prob:.1f}% recorded", fontsize=11, loc="left")
fig.suptitle("Out of every 10 deaths in 2025, how many were recorded?", x=0.02, ha="left", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "06_icon_array.png", dpi=200)
plt.show()

## 7. What changed, and where deaths go unseen

**Verbal autopsies** (structured interviews with families about deaths outside health facilities) rose
tenfold from 2022 to 2025. This **coincided with** the turnaround in death registration; the data cannot show it
caused it. Registration of old deaths also surged (deaths registered over a year late: 328 → 14,517), so the
system is catching up on its backlog as well.

In [ ]:
va_growth = va.loc[2025, "verbal_autopsies"] / va.loc[2022, "verbal_autopsies"]
print(f"verbal autopsies 2022→2025: ×{va_growth:.1f}, average growth {100 * (va_growth ** (1 / 3) - 1):.0f}% a year")
print(backlog[["over_one_year_late"]].T)

fig, ax1 = plt.subplots(figsize=(7.5, 4))
bars = ax1.bar(va.index, va["verbal_autopsies"], color=ACCENT, width=0.6)
ax1.bar_label(bars, labels=[f"{v:,.0f}" for v in va["verbal_autopsies"]], padding=3)
ax1.set_ylabel("Verbal autopsies")
ax1.set_xticks(va.index)
ax1.set_ylim(0, 23_000)
ax2 = ax1.twinx()
ax2.plot(va.index, mortality.loc[2022:, "completeness"], "o-", color=DEATH, lw=2.5)
ax2.set_ylim(0, 60)
ax2.set_ylabel("Deaths recorded (%)", color=DEATH)
ax2.grid(False)
ax2.spines["right"].set_visible(True)
ax1.set_title(f"Verbal autopsies ×{va_growth:.0f} while recorded deaths rose to 50.5%")
fig.tight_layout()
fig.savefig(FIG / "07_verbal_autopsies.png", dpi=200)
plt.show()

### Deaths outside health facilities, by district (2025)
Deaths in the community are the hardest to record, so the districts where most deaths happen outside a facility
are where community reporting matters most.

In [ ]:
d = districts.sort_values("community_pct")
s = d["community_pct"]
print(f"30 districts: mean {s.mean():.1f}%, median {s.median():.1f}%, SD {s.std():.1f}, "
      f"range {s.min():.1f}% ({d.iloc[0]['district']}) – {s.max():.1f}% ({d.iloc[-1]['district']})")
print(f"middle half of districts: {s.quantile(0.25):.1f}% – {s.quantile(0.75):.1f}%")
print(residence[["area", "community_share"]])

fig, (ax_h, ax) = plt.subplots(1, 2, figsize=(11, 7), gridspec_kw={"width_ratios": [1, 2]})
ax_h.hist(s, bins=np.arange(30, 72.5, 2.5), color=DEATH, alpha=0.85, orientation="horizontal")
ax_h.set_ylabel("Deaths outside a health facility (%)")
ax_h.set_xlabel("Number of districts")
ax_h.invert_xaxis()
ax_h.set_title("Distribution")

cmap = plt.get_cmap("OrRd")
ax.barh(d["district"], s, color=cmap(0.25 + 0.75 * (s - s.min()) / (s.max() - s.min())))
ax.axvline(55.9, color=INK, lw=1, ls=":")
ax.text(56.4, 0.2, "Rwanda 55.9%", color=INK, fontsize=9)
for i, v in enumerate(s):
    ax.text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=8, color=MUTED)
ax.set_xlim(0, 75)
ax.grid(axis="y", visible=False)
ax.set_title("Share of registered deaths in the community, by district")
fig.tight_layout()
fig.savefig(FIG / "08_districts.png", dpi=200)
plt.show()

## 8. Key numbers
Everything the poster and the animation quote, in one place.

In [ ]:
key = {
    "birth_completeness_2025": float(y_birth[-1]),
    "death_completeness_2025": float(y_death[-1]),
    "unrecorded_deaths_2025": int(unrecorded),
    "unrecorded_per_day": round(float(unrecorded / 365)),
    "male_completeness_2025": round(100 * float(p_m), 1),
    "female_completeness_2025": round(100 * float(p_f), 1),
    "share_men_all_deaths": round(100 * float(p_man)),
    "share_men_recorded_deaths": round(100 * float(p_man_given_rec)),
    "slope_all_years": round(float(lines["2019–2025 (all years)"][1]), 1),
    "slope_recovery": round(float(lines["2021–2025 (recovery)"][1]), 1),
    "line_90_year": round(float(year_reaching(90, theta_rec))),
    "scurve_90_year": round(float(year_reaching_s(90, k, t0))),
    "scurve_k": round(float(k), 4),
    "scurve_50_year": round(float(BASE + t0), 1),
    "va_2022": int(va.loc[2022, "verbal_autopsies"]),
    "va_2025": int(va.loc[2025, "verbal_autopsies"]),
    "late_deaths_2022": int(backlog.loc[2022, "over_one_year_late"]),
    "late_deaths_2025": int(backlog.loc[2025, "over_one_year_late"]),
    "rural_community_pct": round(100 * float(residence.loc[0, "community_share"]), 1),
    "urban_community_pct": round(100 * float(residence.loc[1, "community_share"]), 1),
}
(FIG / "key_numbers.json").write_text(json.dumps(key, indent=2))
key

## 9. Static infographic (poster)

In [ ]:
fig = plt.figure(figsize=(12, 18))
W = fig.add_axes([0, 0, 1, 1]); W.axis("off"); W.set_xlim(0, 1); W.set_ylim(0, 1)

def heading(x, y, number, text, sub=None):
    W.text(x, y, number, fontsize=13, fontweight="bold", color=DEATH, va="baseline")
    W.text(x + 0.032, y, text, fontsize=17, fontweight="bold", color=INK, va="baseline")
    if sub:
        W.text(x, y - 0.017, sub, fontsize=10.5, color=MUTED, va="top" if "\n" in sub else "baseline")

# ---- 1. Header
W.add_patch(plt.Rectangle((0, 0.915), 1, 0.085, color=INK, lw=0))
W.text(0.05, 0.962, "THE INVISIBLE HALF", fontsize=38, fontweight="bold", color=PAPER, va="center")
W.text(0.05, 0.932, "Rwanda records almost every birth, but only half of its deaths  ·  Vital statistics 2019–2025",
       fontsize=13.5, color="#C9D1D6", va="center")

# ---- Hero: births vs deaths icon arrays
for x0, label, value, filled, color in [(0.05, "of births were recorded in 2025", y_birth[-1], 9, BIRTH),
                                        (0.53, "of deaths were recorded in 2025", y_death[-1], 5, DEATH)]:
    W.text(x0, 0.865, f"{value:.1f}%", fontsize=50, fontweight="bold", color=color, va="center")
    W.text(x0, 0.828, label, fontsize=13, color=INK, va="center")
    icon_row(fig.add_axes([x0 - 0.005, 0.765, 0.42, 0.05]), filled, color=color)
W.text(0.05, 0.742, f"{key['unrecorded_deaths_2025']:,} deaths went unrecorded in 2025: about "
       f"{key['unrecorded_per_day']} every day.", fontsize=14, fontweight="bold", color=DEATH)
W.plot([0.05, 0.95], [0.728, 0.728], color=GRID, lw=1.5)

# ---- 2. Trend + projection
heading(0.05, 0.705, "1", "The gap is closing, fast",
        f"Deaths recorded rose from 26% (2021) to 50.5% (2025). If this pace continues, 90% would be\nrecorded by about "
        f"{key['line_90_year']} (straight line) to {key['scurve_90_year']} (S-curve, fitted by gradient descent).")
ax = fig.add_axes([0.08, 0.475, 0.86, 0.18])
ax.axvspan(2020, 2021, color=GRID, alpha=0.7, lw=0)
ax.text(2020.5, 97, "COVID-19", ha="center", va="top", color=MUTED, fontsize=9)
ax.axhline(90, color=MUTED, lw=1, ls=":")
ax.text(2018.8, 91.5, "90%", ha="left", color=MUTED, fontsize=9)
tt = t_future[t_future >= 6]
ax.plot(BASE + tt, logistic(tt, k, t0), color=DEATH, lw=2.2, ls="--")
ax.plot(BASE + t_future[ok & (t_future >= 6)], line_future[ok & (t_future >= 6)], color=MUTED, lw=1.3, ls="-.")
ax.plot(years, y_birth, "o-", color=BIRTH, lw=3)
ax.plot(years, y_death, "o-", color=DEATH, lw=3)
ax.text(2025.2, 87.5, "Births", color=BIRTH, fontweight="bold", fontsize=11, va="top")
ax.text(2019, 38, "Deaths", color=DEATH, fontweight="bold", fontsize=11)
ax.annotate("50.5%", (2025, 50.5), xytext=(-8, 10), textcoords="offset points", ha="right",
            color=DEATH, fontweight="bold")
ax.text(2030.2, 62, "S-curve projection", color=DEATH, fontsize=9.5, rotation=24)
ax.text(2027.0, 76, "straight line", color=MUTED, fontsize=9, rotation=33)
ax.set_xlim(2018.6, 2035.6); ax.set_ylim(0, 102)
ax.set_xticks(range(2019, 2036, 2))
ax.set_ylabel("% recorded")
W.text(0.08, 0.452, "Rwanda is now at the steepest point of the curve (50%): gains will come more slowly from here.",
       fontsize=10.5, color=INK, style="italic")
W.plot([0.05, 0.95], [0.437, 0.437], color=GRID, lw=1.5)

# ---- 3. What changed: verbal autopsies
heading(0.05, 0.413, "2", "What changed?")
W.text(0.05, 0.396, "Verbal autopsies (family interviews about\ndeaths outside health facilities) grew tenfold.",
       fontsize=10.5, color=MUTED, va="top")
ax = fig.add_axes([0.07, 0.235, 0.38, 0.12])
bars = ax.bar(va.index, va["verbal_autopsies"], color=ACCENT, width=0.62)
ax.bar_label(bars, labels=[f"{v:,.0f}" for v in va["verbal_autopsies"]], padding=3, fontsize=10)
ax.set_xticks(va.index); ax.set_yticks([]); ax.spines["left"].set_visible(False); ax.grid(False)
ax.set_ylim(0, 24_000)
ax.annotate("", xy=(2025, 21_500), xytext=(2022, 4_300),
            arrowprops=dict(arrowstyle="->", color=INK, lw=1.4, connectionstyle="arc3,rad=-0.25"))
ax.text(2022.9, 17_000, "×10", fontsize=22, fontweight="bold", color=INK)
W.text(0.05, 0.208, "This coincided with the 2022–25 rise in recorded deaths.\n"
       f"Old deaths are being registered too: {key['late_deaths_2022']:,} → {key['late_deaths_2025']:,}\n"
       "deaths registered over a year late (2022 → 2025).", fontsize=10, color=INK, va="top")

# ---- 4. Gender gap
heading(0.53, 0.413, "3", "Women's deaths are counted less")
W.text(0.53, 0.396, "Out of every 10 deaths in 2025, recorded:", fontsize=10.5, color=MUTED, va="top")
for yy, lab, prob in [(0.315, "Men", p_m), (0.245, "Women", p_f)]:
    W.text(0.53, yy + 0.047, f"{lab}  {100 * prob:.1f}%", fontsize=13, fontweight="bold", color=INK)
    icon_row(fig.add_axes([0.525, yy, 0.42, 0.04]), round(10 * prob), color=DEATH)
W.text(0.53, 0.225, f"Men are {key['share_men_all_deaths']}% of all deaths but {key['share_men_recorded_deaths']}% of "
       "recorded ones,\nso statistics from registered deaths over-represent men.", fontsize=10, color=INK, va="top")
W.plot([0.05, 0.95], [0.18, 0.18], color=GRID, lw=1.5)

# ---- 5. Districts (compact ranked strip)
heading(0.05, 0.158, "4", "Where deaths happen outside health facilities",
        f"Share of registered deaths in the community, 2025. Rural {key['rural_community_pct']:.0f}% vs urban "
        f"{key['urban_community_pct']:.0f}%. These are the hardest deaths to record.")
ax = fig.add_axes([0.07, 0.097, 0.88, 0.034])
order = districts.sort_values("community_pct", ascending=False).reset_index(drop=True)
vals = order["community_pct"]
ax.bar(range(30), vals, color=cmap(0.25 + 0.75 * (vals - vals.min()) / (vals.max() - vals.min())), width=0.8)
ax.set_xticks(range(30)); ax.set_xticklabels(order["district"], rotation=90, fontsize=7.5)
ax.set_ylim(0, 80); ax.set_yticks([0, 50]); ax.set_yticklabels(["0", "50%"], fontsize=8)
ax.grid(axis="x", visible=False)
for i in (0, 29):
    ax.text(i, vals[i] + 2, f"{vals[i]:.1f}%", ha="center", fontsize=8.5, fontweight="bold", color=INK)

# ---- 6. Policy takeaway + sources
W.add_patch(plt.Rectangle((0, 0), 1, 0.032, color=INK, lw=0))
W.text(0.05, 0.016, "Policy: expand verbal autopsies and community death reporting where most deaths happen outside "
       "facilities, and close the gap for women.", fontsize=10, fontweight="bold", color=PAPER, va="center")
W.text(0.05, 0.036, "Source: NISR, Rwanda Vital Statistics Report 2025 (CRVS), Final Tables_CRVS_2025.xlsx. "
       "Where tables differ, Table 12 values are used (female 2025 = 43.7%, from Table 12 counts).\n"
       "Projections assume the 2021–25 trend continues (7 data points); 'coincided' does not mean 'caused'. "
       "Community = outside a health facility.", fontsize=7.5, color=MUTED, va="bottom")

fig.savefig(FIG / "poster_invisible_half.png", dpi=200)
fig.savefig(FIG / "poster_invisible_half.pdf")
plt.show()

## 10. Dynamic infographic (Plotly animation)
Three acts: the data appears year by year, gradient descent fits the S-curve (with its loss falling),
then the projection extends to 2035.

In [ ]:
proj_years = np.arange(2021, 2035.01, 0.25)
dyn = make_subplots(rows=1, cols=2, column_widths=[0.7, 0.3], horizontal_spacing=0.08,
                    subplot_titles=("Registration completeness (%)", "Gradient descent loss (MSE)"))

def frame_data(n_years, k_, t0_, loss_upto, proj_until):
    yrs = years[:n_years]
    proj = proj_years[proj_years <= proj_until]
    line_y = theta_rec[0] + theta_rec[1] * (proj - BASE)
    show_line = proj_until > 2025.5
    return [
        go.Scatter(x=yrs, y=y_birth[:n_years], mode="lines+markers", name="Births",
                   line=dict(color=BIRTH, width=4), marker=dict(size=9)),
        go.Scatter(x=yrs, y=y_death[:n_years], mode="lines+markers", name="Deaths",
                   line=dict(color=DEATH, width=4), marker=dict(size=9)),
        go.Scatter(x=proj if k_ is not None else [], y=logistic(proj - BASE, k_, t0_) if k_ is not None else [],
                   mode="lines", name="S-curve", line=dict(color=DEATH, width=3, dash="dash")),
        go.Scatter(x=proj[(line_y <= 100) & (proj >= 2025)] if show_line else [],
                   y=line_y[(line_y <= 100) & (proj >= 2025)] if show_line else [],
                   mode="lines", name="Straight line", line=dict(color=MUTED, width=2, dash="dot")),
        go.Scatter(x=np.arange(1, loss_upto + 1), y=loss_hist[:loss_upto], mode="lines",
                   name="Loss", line=dict(color=ACCENT, width=3), showlegend=False),
    ]

frames, labels = [], []
for i in range(1, 8):  # act 1: data appears
    frames.append(go.Frame(data=frame_data(i, None, None, 1, 2021), name=f"a{i}",
                           layout=dict(title_text=f"<b>{years[i-1]}</b>: births {y_birth[i-1]:.1f}%, deaths {y_death[i-1]:.1f}% recorded")))
    labels.append(str(years[i - 1]))
for it in SNAP:  # act 2: the S-curve is fitted
    k_i, t0_i, loss_i = snaps[it]
    frames.append(go.Frame(data=frame_data(7, k_i, t0_i, it + 1, 2025), name=f"g{it}",
                           layout=dict(title_text=f"Gradient descent, iteration {it:,}: MSE = {loss_i:.1f}")))
    labels.append(f"it {it}")
for until in range(2026, 2036):  # act 3: projection
    frames.append(go.Frame(data=frame_data(7, k, t0, len(loss_hist), until), name=f"p{until}",
                           layout=dict(title_text=f"If the 2021–25 pace continues: {logistic(until - BASE, k, t0):.0f}% of deaths recorded in {until}")))
    labels.append(str(until))

for tr, col in zip(frames[0].data, [1, 1, 1, 1, 2]):
    dyn.add_trace(tr, row=1, col=col)
dyn.frames = frames
dyn.add_hline(y=90, line=dict(color=MUTED, width=1, dash="dot"), row=1, col=1)
dyn.update_xaxes(range=[2018.5, 2035.5], dtick=2, row=1, col=1)
dyn.update_yaxes(range=[0, 102], row=1, col=1)
dyn.update_xaxes(type="log", range=[0, np.log10(len(loss_hist))], title_text="iteration", row=1, col=2)
dyn.update_yaxes(type="log", range=[0, np.log10(loss_hist[0] * 1.5)], row=1, col=2)
dyn.update_layout(
    title=dict(text=frames[0].layout.title.text, x=0.02, font=dict(size=20, color=INK)),
    paper_bgcolor=PAPER, plot_bgcolor=PAPER, font=dict(color=INK, family="Arial"),
    height=620, margin=dict(t=110, b=150, l=60, r=30),
    legend=dict(orientation="h", y=1.08, x=0.0),
    updatemenus=[dict(type="buttons", x=0.0, y=-0.13, xanchor="left", direction="left", buttons=[
        dict(label="▶ Play", method="animate",
             args=[None, dict(frame=dict(duration=700, redraw=True), transition=dict(duration=300), fromcurrent=True)]),
        dict(label="❚❚ Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(x=0.15, y=-0.08, len=0.85, currentvalue=dict(visible=False), steps=[
        dict(label=lab, method="animate", args=[[f.name], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
        for f, lab in zip(frames, labels)])],
    annotations=list(dyn.layout.annotations) + [dict(
        text="Source: NISR, Rwanda Vital Statistics Report 2025 (CRVS). Projections assume the 2021–25 trend continues.",
        x=0, y=-0.3, xref="paper", yref="paper", showarrow=False, font=dict(size=11, color=MUTED), xanchor="left")],
)
dyn.write_html(FIG / "dynamic_invisible_half.html", include_plotlyjs=True, auto_play=False)
dyn

## 11. Dynamic infographic as MP4 video (competition format)
NISR accepts the dynamic entry as **MP4 only**, so the same story is rendered as a ~40-second HD video:
title card → data appears year by year → gradient descent fits the S-curve (loss falling) → projection to 2035 → key messages.

In [ ]:
import imageio_ffmpeg
from matplotlib.animation import FuncAnimation, FFMpegWriter

plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
FPS = 20

# parameters at every iteration of the same gradient descent run as section 3
_, _, _, path = gradient_descent(t_fit, y_fit, k=0.5, t0=4.0, keep=set(range(20_000)))
LAST_IT = 19_999
year_90 = year_reaching_s(90, k, t0)
line_90 = year_reaching(90, theta_rec)

ACTS = [("intro", 3.5), ("data", 8), ("data_hold", 1.5), ("fit", 10), ("fit_hold", 1.5),
        ("proj", 8), ("proj_hold", 2.5), ("outro", 6)]
timeline = []
for act, sec in ACTS:
    n_frames = int(sec * FPS)
    timeline += [(act, i / (n_frames - 1)) for i in range(n_frames)]
print(f"{len(timeline)} frames = {len(timeline) / FPS:.0f} s")

vid = plt.figure(figsize=(12.8, 7.2), dpi=100)
ax = vid.add_axes([0.07, 0.13, 0.60, 0.62])
ax_loss = vid.add_axes([0.75, 0.13, 0.22, 0.45])
title = vid.text(0.07, 0.885, "", fontsize=24, fontweight="bold", color=INK)
caption = vid.text(0.07, 0.815, "", fontsize=15, color=INK)
vid.text(0.07, 0.025, "Source: NISR, Rwanda Vital Statistics Report 2025 (CRVS). "
         "Projections assume the 2021–25 trend continues (7 data points).", fontsize=10, color=MUTED)

ax.set_xlim(2018.6, 2035.6); ax.set_ylim(0, 102); ax.set_xticks(range(2019, 2036, 2))
ax.set_ylabel("% of events recorded")
ax.axhline(90, color=MUTED, lw=1, ls=":"); ax.text(2018.8, 91.5, "90%", color=MUTED, fontsize=10)
ax.axvspan(2020, 2021, color=GRID, alpha=0.7, lw=0)
ax.text(2020.5, 98, "COVID-19", ha="center", va="top", color=MUTED, fontsize=9)
birth_line, = ax.plot([], [], "-", color=BIRTH, lw=3.5)
death_line, = ax.plot([], [], "-", color=DEATH, lw=3.5)
birth_dots, = ax.plot([], [], "o", color=BIRTH, ms=8)
death_dots, = ax.plot([], [], "o", color=DEATH, ms=8)
s_line, = ax.plot([], [], color=DEATH, lw=2.5, ls="--")
st_line, = ax.plot([], [], color=MUTED, lw=1.6, ls="-.")
birth_lab = ax.text(2019, 80, "Births", color=BIRTH, fontweight="bold", fontsize=13)
death_lab = ax.text(2019, 37, "Deaths", color=DEATH, fontweight="bold", fontsize=13)
mark_s = ax.annotate(f"90% ≈ {year_90:.0f}\n(S-curve)", (year_90, 90), xytext=(-10, -55), textcoords="offset points",
                     ha="center", color=DEATH, fontweight="bold", fontsize=11,
                     arrowprops=dict(arrowstyle="-", color=DEATH))
mark_l = ax.annotate(f"≈ {line_90:.0f}\n(straight line)", (line_90, 90), xytext=(-40, 18), textcoords="offset points",
                     ha="center", color=MUTED, fontsize=10)

ax_loss.set_xscale("log"); ax_loss.set_yscale("log")
ax_loss.set_xlim(1, 20_000); ax_loss.set_ylim(2, loss_hist[0] * 1.5)
ax_loss.set_title("Gradient descent loss (MSE)", fontsize=12)
ax_loss.set_xlabel("iteration")
loss_line, = ax_loss.plot([], [], color=ACCENT, lw=2.5)

def card_axes():
    c = vid.add_axes([0, 0, 1, 1], zorder=10)
    c.axis("off"); c.set_xlim(0, 1); c.set_ylim(0, 1)
    c.add_patch(plt.Rectangle((0, 0), 1, 1, color=INK))
    return c

intro = card_axes()
intro.text(0.08, 0.62, "THE INVISIBLE HALF", fontsize=54, fontweight="bold", color=PAPER)
intro.text(0.08, 0.50, "Rwanda records almost every birth,\nbut only half of its deaths.", fontsize=26, color="#C9D1D6", va="top")
intro.text(0.08, 0.22, "Vital statistics 2019–2025  ·  NISR CRVS data", fontsize=15, color=ACCENT)

outro = card_axes()
outro.text(0.08, 0.84, f"{y_birth[-1]:.1f}%", fontsize=48, fontweight="bold", color="#6FB7C6")
outro.text(0.08, 0.77, "of births recorded in 2025", fontsize=16, color=PAPER)
outro.text(0.50, 0.84, f"{y_death[-1]:.1f}%", fontsize=48, fontweight="bold", color="#E27A5C")
outro.text(0.50, 0.77, "of deaths recorded in 2025", fontsize=16, color=PAPER)
outro.text(0.08, 0.62, "\n".join([
    f"•  {key['unrecorded_deaths_2025']:,} deaths went unrecorded in 2025: about {key['unrecorded_per_day']} a day.",
    f"•  Verbal autopsies grew ×10 since 2022, as recorded deaths rose.",
    f"•  Men's deaths: {key['male_completeness_2025']}% recorded. Women's: {key['female_completeness_2025']}%.",
    f"•  If the 2021–25 pace continues: 90% recorded by about {key['line_90_year']}–{key['scurve_90_year']}.",
]), fontsize=17, color=PAPER, va="top", linespacing=1.9)
outro.text(0.08, 0.14, "Policy: expand verbal autopsies and community death reporting,\nand close the gap for women.",
           fontsize=17, fontweight="bold", color=ACCENT, va="top")
outro.text(0.08, 0.03, "Source: NISR, Rwanda Vital Statistics Report 2025 (CRVS). Where tables differ, Table 12 values are used.",
           fontsize=10, color="#9AA6AE")

def grow(xs, ys, x_end):
    keep = xs <= x_end
    return np.append(xs[keep], x_end), np.append(ys[keep], np.interp(x_end, xs, ys))

def draw(act, u):
    for name, sec in [("data_hold", "data"), ("fit_hold", "fit"), ("proj_hold", "proj")]:
        if act == name:
            act, u = sec, 1.0
    intro.set_visible(act == "intro")
    outro.set_visible(act == "outro")
    if act in ("intro", "outro"):
        return

    x_end = 2019 + 6 * u if act == "data" else 2025
    for line, dots, ys in [(birth_line, birth_dots, y_birth), (death_line, death_dots, y_death)]:
        line.set_data(*grow(years, ys, x_end))
        dots.set_data(years[years <= x_end], ys[years <= x_end])

    ax_loss.set_visible(act in ("fit", "proj"))
    mark_s.set_visible(act == "proj" and 2025 + 10 * u >= year_90)
    mark_l.set_visible(act == "proj" and 2025 + 10 * u >= line_90)

    if act == "data":
        yr = int(np.floor(x_end))
        i = yr - 2019
        title.set_text("Births are counted. Deaths mostly are not.")
        caption.set_text(f"{yr}:  births {y_birth[i]:.1f}% recorded  ·  deaths {y_death[i]:.1f}% recorded")
        s_line.set_data([], []); st_line.set_data([], []); loss_line.set_data([], [])
    elif act == "fit":
        it = min(int(round(10 ** (u * np.log10(LAST_IT + 1)))) - 1, LAST_IT)
        k_i, t0_i, loss_i = path[it]
        xs = np.linspace(2021, 2025.5, 60)
        s_line.set_data(xs, logistic(xs - BASE, k_i, t0_i))
        st_line.set_data([], [])
        loss_line.set_data(np.arange(1, it + 2), loss_hist[:it + 1])
        title.set_text("Fitting an S-curve with gradient descent")
        caption.set_text(f"iteration {it:,}   ·   k = {k_i:.3f}   ·   error (MSE) = {loss_i:.1f}")
    else:
        x_end = 2025 + 10 * u
        xs = np.linspace(2021, x_end, 200)
        s_line.set_data(xs, logistic(xs - BASE, k, t0))
        xl = np.linspace(2025, min(x_end, year_reaching(100, theta_rec)), 100)
        st_line.set_data(xl, theta_rec[0] + theta_rec[1] * (xl - BASE))
        loss_line.set_data(np.arange(1, len(loss_hist) + 1), loss_hist)
        title.set_text("If the 2021–25 pace continues…")
        caption.set_text(f"{int(x_end)}:  about {logistic(x_end - BASE, k, t0):.0f}% of deaths recorded (S-curve)")

anim = FuncAnimation(vid, lambda f: draw(*timeline[f]), frames=len(timeline), interval=1000 / FPS)
anim.save(FIG / "dynamic_invisible_half.mp4",
          writer=FFMpegWriter(fps=FPS, bitrate=4000, extra_args=["-vcodec", "libx264", "-pix_fmt", "yuv420p"]))
plt.close(vid)
print(f"saved: {(FIG / 'dynamic_invisible_half.mp4').stat().st_size / 1e6:.1f} MB")